# End-to-End Walkthrough: Probabilistic Demand Forecasting

This notebook walks through the full pipeline interactively. See `docs/SRS.md` for the full specification and reasoning behind each step.

Steps:
1. Preprocessing
2. SARIMA baseline
3. LSTM quantile regression model
4. Calibration check
5. Statistical comparison (Diebold-Mariano, Ljung-Box)
6. Newsvendor decision + cost simulation

In [ ]:
import sys
sys.path.append('..')

from src import preprocessing, baseline_sarima, dl_model, calibration, statistical_tests, decision_newsvendor
import numpy as np
import matplotlib.pyplot as plt

## 1. Preprocessing

Make sure your dataset is placed in `data/raw/` and the column names in `src/preprocessing.py` match it before running this cell.

In [ ]:
data = preprocessing.run_preprocessing_pipeline()
X_train, y_train, _ = data['train']
X_val, y_val, _ = data['val']
X_test, y_test, sku_test = data['test']
print(X_train.shape, X_val.shape, X_test.shape)

## 2. SARIMA Baseline

In [ ]:
example_series = X_test[0]
print(baseline_sarima.check_stationarity(example_series))
fitted = baseline_sarima.fit_sarima(example_series)
forecast = baseline_sarima.forecast_sarima(fitted, steps=y_test.shape[1])
print(forecast)

## 3. Train LSTM Quantile Model

See `src/pipeline.py` for the full DataLoader + training loop, reused here for convenience.

In [ ]:
from src.pipeline import build_dataloaders

train_loader, val_loader = build_dataloaders(X_train, y_train, X_val, y_val)
model = dl_model.QuantileLSTM(output_window=y_train.shape[1])
model = dl_model.train_model(model, train_loader, val_loader, epochs=10)

## 4. Predict + Calibration Check

In [ ]:
preds = dl_model.predict(model, X_test)
p10, p50, p90 = preds[:, 0, :], preds[:, 1, :], preds[:, 2, :]

coverage = calibration.empirical_coverage(y_test, p10, p90)
print('Empirical coverage (target 80%):', coverage)

# Plot one example SKU's forecast
sample_idx = 0
calibration.plot_forecast_with_interval(
    dates=range(y_test.shape[1]),
    y_true=y_test[sample_idx],
    median=p50[sample_idx],
    lower=p10[sample_idx],
    upper=p90[sample_idx],
    title='Example Forecast vs Actual'
)

## 5. Statistical Comparison

In [ ]:
dl_errors = (y_test - p50).flatten()
lb_result = statistical_tests.ljung_box_test(dl_errors[:50])
print('Ljung-Box (DL residuals):', lb_result)

# For the Diebold-Mariano test, compute baseline_errors across the SAME
# test set as the DL model (looping SARIMA over all test SKUs), then:
# statistical_tests.diebold_mariano_test(baseline_errors, dl_errors)

## 6. Newsvendor Decision + Cost Simulation

In [ ]:
holding_cost, stockout_cost = 1.0, 4.0
quantile_predictions = {0.1: float(np.mean(p10)), 0.5: float(np.mean(p50)), 0.9: float(np.mean(p90))}
decision = decision_newsvendor.recommend_order_level(quantile_predictions, holding_cost, stockout_cost)
print(decision)

cost_result = decision_newsvendor.simulate_cost(
    decision['recommended_order_level'], y_test.flatten(), holding_cost, stockout_cost
)
print(cost_result)